In [ ]:
import numpy as np
from vpython import *

class Charge:
    def __init__(self, q: float, r: np.ndarray, v: np.ndarray = np.array([0.,0.,0.]), m: float = 1) -> None:
        self.q = q
        self.m = m
        self.rp = r
        self.vp = v

    def E(self, X , Y, Z, eps=1e-4):
        dx = X - self.rp[0]
        dy = Y - self.rp[1]
        dz = Z - self.rp[2]
        d3 = (dx**2 + dy**2 + dz**2 + eps**2)**1.5
        return self.q*np.array([ dx/d3, dy/d3, dz/d3  ])

    def phi(self, X, Y, Z, eps=1e-4):
        dx = X - self.rp[0]
        dy = Y - self.rp[1]
        dz = Z - self.rp[2]
        d3 = (dx**2 + dy**2 + dz**2 + eps**2)**0.5
        return self.q/d3

# distribución de carga que genera el campo electrico
Q = [
    Charge(1, np.array([0, 0, -1.0])),
    Charge(-1, np.array([0, 0, 1.0])),
]

# carga de prueba (la que se va a mover)
q = Charge(-1,
           np.array([0.0, 1.0, 1.0]),
           np.array([-1.0, -1.0, 0.0])
)

scene = canvas(width=600, height=400)

# dibujo de las cargas correspondientes a las distribución de carga
pQ = [
    sphere(pos=vec(*qi.rp), radius=0.1, color=color.red if qi.q > 0 else color.blue)
    for qi in Q
]

# dibujo de la carga de prueba
pq = sphere(pos=vec(*q.rp), radius=0.1, color=color.red if q.q > 0 else color.blue, make_trail = True)


# discretización espacial
nx = 15
x = np.linspace(-2, 2, nx)
y = np.linspace(-2, 2, nx)
z = np.linspace(-2, 2, nx)
X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

L = 0.2
# Indice de x = 0
idx = np.argmin(np.abs(x))
# Indice de y = 0
idy = np.argmin(np.abs(y))

# principio de superposición del campo eléctrico (distribucion de carga inmovil)
E = sum( qi.E(X, Y, Z) for qi in Q )
Emag = np.linalg.norm( E, axis=0 )

#dibujar las lineas de campo de las cargas inmoviles en los planos yz y xz
for i in range(nx):
    for j in range(nx):
        # slice x = 0  (plano yz). Fijar el primer índice
        e = E[:, idx, i, j]
        arrow(
            pos = vec(0, Y[idx, i, j], Z[idx, i, j]),
            axis = L*vec(*e)/Emag[idx, i, j],
            shaftwidth=0.02,
        )

        # slice y = 0  (plano xz). Fijar el segundo índice
        e = E[:, i, idy, j]
        arrow(
            pos = vec(X[i, idy, j], 0, Z[i, idy, j]),
            axis = L*vec(*e)/Emag[i, idy, j],
            shaftwidth = 0.02,
        )

dt = 1e-3
t = 0.0

for k in range(2000):
    rate(500)

    Eq = sum(qi.E(*q.rp) for qi in Q)
    q.vp = q.vp + q.q/q.m*Eq*dt
    q.rp = q.rp + q.vp*dt
    t += dt

    pq.pos = vec(*q.rp)
